# STBSR on Human DLPFC section 151672

Train STBSR from raw counts and spatial coordinates. Evaluate both KMeans readouts using the reference annotations.

## Configure inputs

Place the original Visium counts, tissue positions and `151672_truth.txt` under `1.DLPFC/151672`. Set `data_root` in the next cell or set `STBSR_DATA_ROOT` before opening Jupyter.

In [ ]:
import json
import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

here = Path.cwd().resolve()
repo = next((parent for parent in (here, *here.parents)
             if (parent / "config.json").is_file()
             and (parent / "STBSR" / "pipeline.py").is_file()), None)
if repo is None:
    raise FileNotFoundError("Open Jupyter from the repository or its notebooks directory.")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from STBSR.cli import train_or_predict
from STBSR.reproduction import display_labels, evaluate_file, plot_domains, read_reference

config_path = repo / "config.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
data_root = Path(os.environ.get("STBSR_DATA_ROOT", repo / "data")).expanduser()
device = os.environ.get("STBSR_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dataset = '151672'
seed = 11059586
spot_size = 4
spec = config["dataset_specs"][dataset]
source = data_root / spec["path"]
if not source.exists():
    raise FileNotFoundError(f"Set data_root to the original datasets. Missing: {source}")
print(f"{dataset} | K={spec['given_k']} | seed={seed} | {device}")

## Train STBSR

Training uses the same SCE, BCE and CCR objective from the first update to the stopping point.

In [ ]:
from tempfile import mkdtemp
from types import SimpleNamespace

output_root = repo / "outputs"
output_root.mkdir(exist_ok=True)
output = Path(mkdtemp(prefix=f"{dataset}_seed{seed}_", dir=output_root))
replay_device = config["benchmark_replay_devices"].get(dataset, device) if device == "cuda" else "cpu"

train_or_predict(SimpleNamespace(
    action="train", config=config_path, dataset=dataset, input=source,
    device=device, seed=seed, replay_device=replay_device, out=output,
    compact=True, save_reconstruction=False,
))

run = json.loads((output / "run.json").read_text(encoding="utf-8"))
print(f"Updates: {run['completed_updates']} | Plateau reached: {run['converged']}")
losses = pd.read_csv(output / "loss.csv")
ax = losses.plot(x="CompletedSteps", y=["Loss", "SCE", "BCE", "CCR"], figsize=(7, 3))
ax.set(xlabel="Training update", ylabel="Loss")
plt.show()

## Evaluate and visualize spatial domains

The reference annotations are read after training. Colours align clusters to the reference for display.

In [ ]:
reference = read_reference(dataset, spec, source)
prediction_file = output / "predictions.npz"
scores = pd.DataFrame(evaluate_file(prediction_file, reference)).set_index("readout")
display(scores[["n_scored", "ARI", "NMI", "BF", "BandAcc"]].round(4))

with np.load(prediction_file, allow_pickle=False) as data:
    coords = data["coordinates"]
    frame = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1]})
    frame["reference"] = reference.reindex(data["barcodes"]).to_numpy()
    readouts = {
        "64-dimensional embedding": "embedding64",
        "Normalized reconstruction": "reconstruction_unit",
    }
    assignments = {
        f"{title} | ARI {scores.loc[key, 'ARI']:.4f}":
            display_labels(frame["reference"], pd.Series(data[key], index=frame.index))
        for title, key in readouts.items()
    }
fig = plot_domains(frame, assignments, size=spot_size)
plt.show()

## Compare with Supplementary Data 1

The normalized reconstruction is the readout reported in Fig. 2c.

In [ ]:
reported = pd.read_csv(repo / "benchmark" / "stbsr_per_run.csv", dtype={"dataset": str})
reported = reported.loc[reported["dataset"].eq(dataset), ["run", "seed", "ARI"]]
display(reported.round(4))
print(f"Reported reconstruction ARI over {len(reported)} runs: "
      f"{reported.ARI.mean():.4f} ± {reported.ARI.std():.4f} (sample SD)")